# Needlet / Harmonic ILC Compton-$y$ on FLAMINGO mocks (pyILC)

McCarthy & Hill (2024) style undeprojected Compton-$y$ ILC
([arXiv:2307.01043](https://doi.org/10.48550/arxiv.2307.01043)) on **FLAMINGO mock**
skies (lensed CMB + tSZ + kSZ + CIB) **plus Planck NPIPE noise splits A/B** at 100/143/353 GHz.

This notebook is a thin driver — the pipeline lives in the `flamingo_mock.ilc` package
(`prepare` → `config` → `run` → `validate`, also available as the `flamingo-ilc` command).

Settings:
- `ELLMAX: 3000`, `N_side: 2048`
- Channel beams 9.66′ / 7.22′ / 4.92′; common ILC beam 5′ (spectra beam-deconvolved)
- `ILC_preserved_comp: tSZ`, `N_deproj: 0`
- NPIPE detector-set splits A/B (`mc_00200`), independently destriped → A×B is noise-decoupled
- ILC weight solves on the **JAX backend (GPU)** by default

**Environment:** `source /scratch/scratch-lxu/venv/cmbagent_env/bin/activate`

In [ ]:
import os
from pathlib import Path

REPO = Path('..').resolve()
os.chdir(REPO)

from pyilc.ilc_linalg import available_backends
from flamingo_mock.ilc.paths import ILCPaths

paths = ILCPaths(nside=2048)
print('cwd', Path.cwd())
print('inputs', paths.inputs_dir)
print('backends', available_backends())

## 1. Prepare beam-smoothed coadds + NPIPE noise splits (nside=2048)

In [ ]:
!flamingo-ilc prepare --nside 2048

## 2. Write pyILC YAML configs (tracked artifacts in `configs/`)

In [ ]:
!flamingo-ilc config --out-dir configs --backend jax

## 3. HILC on noise splits A and B (ELLMAX=3000, JAX on GPU)

In [ ]:
!flamingo-ilc run configs/hilc_y_flamingo_npipe_splitA.yml
!flamingo-ilc run configs/hilc_y_flamingo_npipe_splitB.yml

## 4. Optional: paper-style NILC (Gaussian needlets)

In [ ]:
# Heavier than HILC; intermediates under ilc/nilc_output_npipe_splitA/
# !flamingo-ilc run configs/nilc_y_flamingo_npipe_splitA.yml

## 5. Validate (beam-deconvolved $C_\ell$ to $\ell=3000$)

In [ ]:
yA = paths.output_dir('hilc', 'A') / 'flamingo_needletILCmap_component_tSZ_hilc_y_npipe_splitA.fits'
yB = paths.output_dir('hilc', 'B') / 'flamingo_needletILCmap_component_tSZ_hilc_y_npipe_splitB.fits'
assert yA.is_file(), yA
!flamingo-ilc validate --ymap {yA} --ymap-split {yB} --truth {paths.truth_map()} --figures-dir figures --lmax 3000 --ilc-beam-fwhm-arcmin 5.0